In [ ]:
"""
Kikuyu <-> English Translation — Premium Minimalist Interface
Model: NjeriKahoro/nllb-200-Thiomi-Kik-Eng
"""

import torch
import gradio as gr
from transformers import AutoModelForSeq2SeqLM, NllbTokenizer

In [ ]:
# ── Config
MODEL_NAME   = "NjeriKahoro/nllb-200-Thiomi-Kik-Eng-3"
LANG_KI_CODE = "kik_Latn"
LANG_EN_CODE = "eng_Latn"
MAX_LENGTH   = 128

# Force CPU if running on free Hugging Face infrastructure without GPU hardware
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
print(f"Loading model on {DEVICE}...")
tokenizer = NllbTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.config.forced_bos_token_id = None
model.eval()

In [ ]:
# ── Translation Logic
def translate(text: str, src_lang: str, tgt_lang: str) -> str:
    text = text.strip()
    if not text:
        return ""
    tokenizer.src_lang = src_lang
    inputs  = tokenizer(text, return_tensors="pt", truncation=True,
                        max_length=MAX_LENGTH, padding=True).to(DEVICE)
    bos_id  = tokenizer.convert_tokens_to_ids(tgt_lang)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            forced_bos_token_id=bos_id,
            max_length=MAX_LENGTH,
            num_beams=4,
            length_penalty=1.0,
            early_stopping=True,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [ ]:
def run_translation(text: str, direction: str) -> str:
    if not text.strip():
        return "Please enter some text."
    try:
        src, tgt = (LANG_KI_CODE, LANG_EN_CODE) if direction == "Kikuyu to English" \
                   else (LANG_EN_CODE, LANG_KI_CODE)
        return translate(text, src, tgt)
    except Exception as e:
        return f"Error: {e}"

In [ ]:
# ── Examples
examples = [
    ["Nĩ mwega mũno.",                "Kikuyu to English"],
    ["Rũciũ rũrĩa nĩ rũega.",           "Kikuyu to English"],
    ["Mbura nene na riũwa inini?",       "Kikuyu to English"],
    ["Nĩ ũhoro waku atĩa?",             "Kikuyu to English"],
    ["It's a good day today.",           "English to Kikuyu"],
    ["How are you doing?",               "English to Kikuyu"],
    ["The child is playing outside.",    "English to Kikuyu"],
    ["Where are you going?",             "English to Kikuyu"],
]

# Premium Minimalist UI Styles - Accent Color Removed from selection labels
custom_css = """
body, html { background-color: #f8fafc !important; }
.gradio-container { max-width: 1000px !important; font-family: -apple-system, sans-serif !important; }
h1 { font-family: -apple-system, sans-serif !important; letter-spacing: -0.03em; color: #0f172a !important; }
.direction-selector { background: transparent !important; border: none !important; box-shadow: none !important; }
.direction-selector .wrap { justify-content: center !important; gap: 24px !important; }
textarea { border-radius: 12px !important; border: 1px solid #e2e8f0 !important; font-size: 16px !important; background-color: #ffffff !important; padding: 18px !important; color: #334155 !important; }
textarea:focus { border-color: #4f46e5 !important; box-shadow: 0 0 0 4px rgba(79, 70, 229, 0.08) !important; }
.primary-btn { background-color: #4f46e5 !important; color: #ffffff !important; font-weight: 500 !important; border-radius: 8px !important; border: none !important; }
.primary-btn:hover { background-color: #4338ca !important; }
.secondary-btn { background-color: #ffffff !important; color: #475569 !important; border: 1px solid #cbd5e1 !important; border-radius: 8px !important; }
.secondary-btn:hover { background-color: #f1f5f9 !important; }
.gr-form + div > span { color: #059669 !important; font-weight: 600 !important; }
"""

# ── UI Construction
with gr.Blocks(title="Translation Studio", theme=gr.themes.Soft(primary_hue="indigo", radius_size="md"), css=custom_css) as demo:

    gr.HTML("""
    <div style="text-align: center; padding: 48px 0 32px 0;">
        <h1 style="font-size: 2rem; font-weight: 700; color: #0f172a; margin: 0 0 6px 0;">
            Translation Studio
        </h1>
        <p style="color: #64748b; font-size: 1rem; font-weight: 400; margin: 0;">
            High-fidelity Kikuyu and English text localization
        </p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            direction = gr.Radio(
                choices=["Kikuyu to English", "English to Kikuyu"],
                value="Kikuyu to English",
                label="Select language pathway",
                interactive=True,
                elem_classes="direction-selector"
            )

    with gr.Row(equal_height=True):
        input_text = gr.Textbox(
            label="Original text",
            placeholder="Type or paste cross-lingual sentences here...",
            lines=8,
            show_label=True,
        )
        output_text = gr.Textbox(
            label="Localized target text",
            lines=8,
            interactive=False,
            show_label=True,
        )

    with gr.Row():
        clear_btn     = gr.Button("Clear", variant="secondary", elem_classes="secondary-btn")
        translate_btn = gr.Button("Translate", variant="primary", scale=2, elem_classes="primary-btn")

    gr.Examples(
        examples=examples,
        inputs=[input_text, direction],
        outputs=output_text,
        fn=run_translation,
        cache_examples=False,
        label="Reference Expressions",
    )

    gr.HTML("""
    <div style="margin-top: 48px; padding: 20px 0; border-top: 1px solid #e2e8f0; text-align: center;">
        <p style="font-size: 0.85rem; color: #94a3b8; font-weight: 400; margin: 0;">
            Processing standard: NLLB Sequence-to-Sequence (Beam 4) &middot; Context depth: 128 tokens
        </p>
    </div>
    """)

    translate_btn.click(fn=run_translation,     inputs=[input_text, direction], outputs=output_text)
    input_text.submit(fn=run_translation,        inputs=[input_text, direction], outputs=output_text)
    clear_btn.click(fn=lambda: ("", ""),         outputs=[input_text, output_text])

demo.launch()